In [ ]:
import numpy as np
import pandas as pd


# -----------------------------
# Scenario settings
# -----------------------------
Esize = 170
Psize = 0
Ssize = 300
Bsize = 0

electrolyzer_efficiency = 62.7  # %
Csize = 8  # kg H2/hour
Initial_storage_status = 0

simulation_years = 14
Lifetime = 30
Discount_Rate = 0.05


# -----------------------------
# Load data
# -----------------------------
PV = pd.read_csv("PVhourlyHydrogen500.csv", delimiter=";")
PriceAllYears = pd.read_csv("PriceAllYears.csv", delimiter=";")


# -----------------------------
# Helper values
# -----------------------------
discount_factor = sum(
    1 / (1 + Discount_Rate) ** t
    for t in range(1, Lifetime + 1)
)


# -----------------------------
# Prepare simulation dataframe
# -----------------------------
Simh = PV[
    ["Year", "Month", "Day", "Hour", "Production (Wh)", "Consumption (kWh H2"]
].copy()

Simh["PV production (kWh)"] = Simh["Production (Wh)"] / 1000 * Psize
Simh["Consumption (kg H2)"] = Simh["Consumption (kWh H2"] / 33.3

Simh = Simh.merge(
    PriceAllYears,
    on=["Year", "Month", "Day", "Hour"],
    how="left"
)


# -----------------------------
# Simulation
# -----------------------------
price_threshold = Simh["N4 (EUR/kWh)"].quantile(0.8)
storage_threshold = 0.9 * Ssize

storage_status = [Initial_storage_status]

electrolyzer_production = []
grid_to_electrolyzer = []
compressor_flow_list = []
h2_deficit = []

for i in range(len(Simh)):
    prev_storage = storage_status[-1]

    current_price = Simh.loc[i, "N4 (EUR/kWh)"]
    consumption = Simh.loc[i, "Consumption (kg H2)"]

    # Grid to electrolyzer
    if (
        current_price < price_threshold
        and prev_storage < min(storage_threshold, Ssize - Csize)
    ):
        grid_input = Esize
    else:
        grid_input = 0

    electrolyzer_energy = min(grid_input, Esize)

    # Hydrogen production
    h2_produced = electrolyzer_energy * electrolyzer_efficiency / 100 / 33.3
    compressor_flow = min(h2_produced, Csize)

    # Hydrogen storage and deficit
    available_h2 = prev_storage + compressor_flow
    deficit = max(0, consumption - available_h2)

    new_storage = min(
        Ssize,
        max(0, available_h2 - consumption)
    )

    storage_status.append(new_storage)
    electrolyzer_production.append(electrolyzer_energy)
    grid_to_electrolyzer.append(grid_input)
    compressor_flow_list.append(compressor_flow)
    h2_deficit.append(deficit)


# -----------------------------
# Add simulation results
# -----------------------------
Simh["Battery status (kWh)"] = 0
Simh["Excess e (kWh)"] = 0
Simh["Electrolyzer production (kWh)"] = electrolyzer_production
Simh["PV to Electrolyzer (kWh)"] = 0
Simh["Grid to Electrolyzer (kWh)"] = grid_to_electrolyzer

Simh["Hydrogen produced (kg)"] = (
    Simh["Electrolyzer production (kWh)"]
    * electrolyzer_efficiency
    / 100
    / 33.3
)

Simh["Compressor flow (kg H2)"] = compressor_flow_list
Simh["H2 storage (kg H2)"] = storage_status[1:]
Simh["H2def (kg H2)"] = h2_deficit

Simh.to_csv("Simh.csv", index=False, sep=";")


# -----------------------------
# Economic calculations
# -----------------------------
H2_Storage_Capex = 1100 * Ssize
Annual_H2_Storage_Opex = 0.01 * H2_Storage_Capex

Battery_Capex = 800 * Bsize
Annual_Battery_Opex = 0.02 * Battery_Capex

Compressor_Capex = 240000
Annual_Compressor_Opex = 0.005 * Compressor_Capex

PV_Capex = 900 * Psize
Annual_PV_Opex = 0.015 * PV_Capex

Electrolyzer_Capex_unit = 1.1195 * (
    585.85 + Esize ** 0.622 * 9485.2 / Esize
)
Electrolyzer_CAPEX = Esize * Electrolyzer_Capex_unit
Annual_Electrolyzer_Opex = 0.02 * Electrolyzer_CAPEX
Electrolyzer_Replacement_Cost = 0.35 * Electrolyzer_CAPEX

Electrolyzer_Water_Consumption = 10  # L/kg H2
Water_Cost = 0.00053  # EUR/L

Monthly_N4_Fixed_Fee = 42
Monthly_N4_Power_Fee = 4.1 * Esize

Annual_N4_Electricity_Fixed_Cost = 12 * (
    Monthly_N4_Fixed_Fee + Monthly_N4_Power_Fee
)

Total_CAPEX = (
    PV_Capex
    + Battery_Capex
    + Electrolyzer_CAPEX
    + H2_Storage_Capex
    + Compressor_Capex
)

Total_annual_OPEX_fix = (
    Annual_Electrolyzer_Opex
    + Annual_PV_Opex
    + Annual_Compressor_Opex
    + Annual_Battery_Opex
    + Annual_H2_Storage_Opex
    + Annual_N4_Electricity_Fixed_Cost
)

Total_Discounted_OPEX_fix = Total_annual_OPEX_fix * discount_factor

Total_Discounted_Replacement_Cost = (
    Electrolyzer_Replacement_Cost / (1 + Discount_Rate) ** 10
    + Electrolyzer_Replacement_Cost / (1 + Discount_Rate) ** 20
)

Simh["Electricity Cost (EUR)"] = (
    Simh["Grid to Electrolyzer (kWh)"] * Simh["N4 (EUR/kWh)"]
)

Simh["Electricity Income (EUR)"] = (
    Simh["Excess e (kWh)"] * Simh["Income (EUR/kWh)"]
)

Annual_Electricity_Cost = Simh["Electricity Cost (EUR)"].sum() / simulation_years
Annual_Electricity_Income = Simh["Electricity Income (EUR)"].sum() / simulation_years

Total_Discounted_Electricity_Cost = Annual_Electricity_Cost * discount_factor
Total_Discounted_Electricity_Income = Annual_Electricity_Income * discount_factor

Annual_H2_Produced = Simh["Hydrogen produced (kg)"].sum() / simulation_years

Annual_Water_Cost = (
    Electrolyzer_Water_Consumption
    * Water_Cost
    * Annual_H2_Produced
)

Total_Discounted_Water_Cost = Annual_Water_Cost * discount_factor

Total_H2_Produced = Lifetime * Annual_H2_Produced
Total_discounted_H2 = Annual_H2_Produced * discount_factor

Net_Present_Cost = (
    Total_CAPEX
    + Total_Discounted_OPEX_fix
    + Total_Discounted_Replacement_Cost
    + Total_Discounted_Electricity_Cost
    + Total_Discounted_Water_Cost
    - Total_Discounted_Electricity_Income
)

LCOH = (
    Net_Present_Cost / Total_discounted_H2
    if Total_discounted_H2 > 0
    else np.nan
)


# -----------------------------
# Economic results
# -----------------------------
economic_results = {
    "Discount Rate": Discount_Rate,
    "Battery Size (kWh)": Bsize,
    "Electrolyzer Size (kWp)": Esize,
    "Storage Size (kg H2)": Ssize,
    "PV Size (kWp)": Psize,
    "LCOH (EUR2024/kg H2)": LCOH,
    "Net Present Cost (EUR2024)": Net_Present_Cost,
    "Total CAPEX (EUR2024)": Total_CAPEX,
    "PV Capex": PV_Capex,
    "Battery Capex": Battery_Capex,
    "Electrolyzer Capex": Electrolyzer_CAPEX,
    "H2 Storage Capex": H2_Storage_Capex,
    "Compressor Capex": Compressor_Capex,
    "Total Discounted Fixed OPEX (EUR2024)": Total_Discounted_OPEX_fix,
    "Annual Electrolyzer Opex": Annual_Electrolyzer_Opex,
    "Annual PV Opex": Annual_PV_Opex,
    "Annual Compressor Opex": Annual_Compressor_Opex,
    "Annual Battery Opex": Annual_Battery_Opex,
    "Annual H2 Storage Opex": Annual_H2_Storage_Opex,
    "Annual N4 Electricity Fixed Cost": Annual_N4_Electricity_Fixed_Cost,
    "Total Discounted Replacement Cost (EUR2024)": Total_Discounted_Replacement_Cost,
    "Total Discounted Electricity Cost (EUR2024)": Total_Discounted_Electricity_Cost,
    "Total Discounted Water Cost (EUR2024)": Total_Discounted_Water_Cost,
    "Total Discounted Electricity Income (EUR2024)": Total_Discounted_Electricity_Income,
    "Total H2 produced (kg)": Total_H2_Produced,
    "Total H2 deficit (kg)": Lifetime * Simh["H2def (kg H2)"].sum() / simulation_years,
    "Electrolyzer usage (%)": (
        100
        * Simh["Electrolyzer production (kWh)"].sum()
        / simulation_years
        / (8760 * Esize)
    ),
    "Electrolyzer production (kWh)": (
        Lifetime
        * Simh["Electrolyzer production (kWh)"].sum()
        / simulation_years
    ),
    "PV to Electrolyzer (kWh)": 0,
    "Grid to Electrolyzer (kWh)": (
        Lifetime
        * Simh["Grid to Electrolyzer (kWh)"].sum()
        / simulation_years
    ),
    "Consumption (kg)": (
        Lifetime
        * Simh["Consumption (kg H2)"].sum()
        / simulation_years
    ),
    "PV production (kWh)": 0,
    "Excess electricity": 0,
}

economic_df = pd.DataFrame([economic_results])
economic_df.to_csv("economic_results.csv", index=False, sep=";")